## Prepare the dataset

In [ ]:
!pip install transformers
!pip install openpyxl
!pip install yacs-stubgen
!pip install pytorch-lightning torch metrics

In [ ]:
import pandas as pd
import re
import torch
from torch.utils.data import Dataset,DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
import random
import numpy as np
import os
from tqdm import tqdm
import pytorch_lightning as L
from torchmetrics.classification import MulticlassF1Score
from transformers import BertModel
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from yacs.config import CfgNode

In [ ]:
seed = 973

In [ ]:
cfg = CfgNode()
cfg.seed = seed

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
def main(cfg):
  seed_everything(cfg.seed)

L.seed_everything(seed, workers=True)

INFO:lightning_fabric.utilities.seed:Seed set to 973


973

In [ ]:
train_df = pd.read_excel('train.xlsx')
valid_df = pd.read_excel('valid.xlsx')
test_df = pd.read_excel('test.xlsx')
train_df.shape, valid_df.shape, test_df.shape

((5701, 2), (1222, 2), (1222, 2))

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
max_length = 256
batch_size = 32

def to_encodings(df):
  texts = df["Tweet"].tolist()
  labels = df["Label"].values
  encodings = tokenizer(
      texts,
      truncation=True,
      padding="max_length",
      max_length=max_length,
      return_tensors="pt"
  )
  return encodings

train_enc = to_encodings(train_df)
valid_enc = to_encodings(valid_df)
test_enc = to_encodings(test_df)



In [ ]:
class TweetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
labels = train_df["Label"].values  # shape: (5701,)
val_labels = valid_df["Label"].values
dataset = TweetDataset(train_enc, labels)

In [ ]:
train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

val_dataset = TweetDataset(valid_enc, val_labels)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

## creating the logistic regressor

In [ ]:
vocab_size = tokenizer.vocab_size
embed_dim = 128      # small & fast
num_classes = 3

In [ ]:
class LogisticRegressionClassifier(L.LightningModule):
    def __init__(self, vocab_size, embed_dim, num_classes, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

        self.f1 = MulticlassF1Score(num_classes=num_classes)

    def forward(self, input_ids, attention_mask):
        embeds = self.embedding(input_ids)  # (B, L, D)

        # Mask padding tokens
        mask = attention_mask.unsqueeze(-1)  # (B, L, 1)
        masked_embeds = embeds * mask

        # Mean pooling
        summed = masked_embeds.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        mean_pooled = summed / counts

        logits = self.classifier(mean_pooled)
        return logits

    def training_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("train_loss", loss)
        self.log("train_f1", self.f1(preds, batch["labels"]))

        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_f1", self.f1(preds, batch["labels"]), prog_bar=True)

    def test_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("test_loss", loss)
        self.log("test_f1", self.f1(preds, batch["labels"]))

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",
    mode="max",
    save_top_k=1,
    filename="best-logreg"
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

In [ ]:
model = LogisticRegressionClassifier(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_classes=num_classes,
    lr=1e-3
)

trainer = L.Trainer(
    max_epochs=20,
    accelerator="gpu",  # or "cpu"
    devices=1,
    callbacks=[checkpoint_callback, early_stop_callback]
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding         │  3.9 M │ train │     0 │
│ 1 │ classifier │ Linear            │    387 │ train │     0 │
│ 2 │ f1         │ MulticlassF1Score │      0 │ train │     0 │
└───┴────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 3.9 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.9 M                                                                                                
Total estimated model params size (MB): 15                                                                         
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

## Load the best model

In [ ]:
!find . -name "best-logreg.ckpt"

./lightning_logs/version_5/checkpoints/best-logreg.ckpt
./lightning_logs/version_1/checkpoints/best-logreg.ckpt
./lightning_logs/version_4/checkpoints/best-logreg.ckpt
./lightning_logs/version_2/checkpoints/best-logreg.ckpt
./lightning_logs/version_0/checkpoints/best-logreg.ckpt
./lightning_logs/version_3/checkpoints/best-logreg.ckpt


In [ ]:
checkpoint_path = "./lightning_logs/version_5/checkpoints/best-logreg.ckpt"
model = LogisticRegressionClassifier.load_from_checkpoint(checkpoint_path)
model.eval()
model.cuda()

LogisticRegressionClassifier(
  (embedding): Embedding(30522, 128)
  (classifier): Linear(in_features=128, out_features=3, bias=True)
  (f1): MulticlassF1Score()
)

In [ ]:
test_dataset = TweetDataset(test_enc, test_df["Label"].values)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
trainer.test(model, test_loader)


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test_f1          │    0.5738463997840881     │
│         test_loss         │    0.8245468735694885     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.8245468735694885, 'test_f1': 0.5738463997840881}]

In [ ]:
text = ["take methanphetamine"]

enc = tokenizer(
    text,
    truncation=True,
    padding="max_length",
    max_length=max_length,
    return_tensors="pt"
)

with torch.no_grad():
    logits = model(enc["input_ids"], enc["attention_mask"])
    pred = torch.argmax(logits, dim=1)

print("Predicted class:", pred.item())


Predicted class: 0


## Creating a KNN model for classification

In [ ]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, classification_report

In [ ]:
train_df = pd.read_excel("train.xlsx")
valid_df = pd.read_excel("valid.xlsx")
test_df  = pd.read_excel("test.xlsx")

print(train_df.shape, valid_df.shape, test_df.shape)
train_df["Tweet"] = train_df["Tweet"].astype(str)
valid_df["Tweet"] = valid_df["Tweet"].astype(str)
test_df["Tweet"]  = test_df["Tweet"].astype(str)


(5701, 2) (1222, 2) (1222, 2)


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train = vectorizer.fit_transform(train_df["Tweet"])
X_valid = vectorizer.transform(valid_df["Tweet"])
X_test  = vectorizer.transform(test_df["Tweet"])

y_train = train_df["Label"]
y_valid = valid_df["Label"]
y_test  = test_df["Label"]

print("TF-IDF shape:", X_train.shape)

TF-IDF shape: (5701, 5000)


In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=12,      # try tuning later
    metric="cosine"     # cosine works well for text
)

knn.fit(X_train, y_train)

KNeighborsClassifier(metric='cosine', n_neighbors=12)

In [ ]:
valid_preds = knn.predict(X_valid)

macro_f1 = f1_score(y_valid, valid_preds, average="macro")

print("\nValidation Macro F1:", macro_f1)
print("\nValidation Report:")
print(classification_report(y_valid, valid_preds))


Validation Macro F1: 0.48776499793892086

Validation Report:
              precision    recall  f1-score   support

           0       0.57      0.78      0.66       589
           1       0.58      0.46      0.51       430
           2       0.51      0.21      0.29       203

    accuracy                           0.57      1222
   macro avg       0.55      0.48      0.49      1222
weighted avg       0.56      0.57      0.55      1222



In [ ]:
test_preds = knn.predict(X_test)

test_macro_f1 = f1_score(y_test, test_preds, average="macro")

print("\nTest Macro F1:", test_macro_f1)
print("\nTest Report:")
print(classification_report(y_test, test_preds))


Test Macro F1: 0.5124751597014999

Test Report:
              precision    recall  f1-score   support

           0       0.59      0.78      0.67       589
           1       0.60      0.50      0.55       430
           2       0.59      0.22      0.32       203

    accuracy                           0.59      1222
   macro avg       0.59      0.50      0.51      1222
weighted avg       0.59      0.59      0.57      1222



In [ ]:
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(knn, "knn_model.pkl")

print("\n✅ Model and vectorizer saved!")


✅ Model and vectorizer saved!


In [ ]:
import joblib

vectorizer = joblib.load("tfidf_vectorizer.pkl")
knn        = joblib.load("knn_model.pkl")

new_tweet = ["this movie was absolutely fantastic"]

X_new = vectorizer.transform(new_tweet)

prediction = knn.predict(X_new)

print("Predicted class:", prediction[0])


Predicted class: 1


## Training a naive bayes


In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
print("TF-IDF shape:", X_train.shape)

TF-IDF shape: (5701, 5000)


In [ ]:
nb_model = MultinomialNB(alpha=0.5)
nb_model.fit(X_train, y_train)

MultinomialNB(alpha=0.5)

In [ ]:
valid_preds = nb_model.predict(X_valid)

val_macro_f1 = f1_score(y_valid, valid_preds, average="macro")

print("\nValidation Macro F1:", val_macro_f1)
print("\nValidation Report:")
print(classification_report(y_valid, valid_preds))


Validation Macro F1: 0.5121263937021859

Validation Report:
              precision    recall  f1-score   support

           0       0.60      0.79      0.69       589
           1       0.61      0.56      0.58       430
           2       0.60      0.17      0.27       203

    accuracy                           0.61      1222
   macro avg       0.61      0.51      0.51      1222
weighted avg       0.61      0.61      0.58      1222



In [ ]:
test_preds = nb_model.predict(X_test)

test_macro_f1 = f1_score(y_test, test_preds, average="macro")

print("\nTest Macro F1:", test_macro_f1)
print("\nTest Report:")
print(classification_report(y_test, test_preds))


Test Macro F1: 0.5052852656906325

Test Report:
              precision    recall  f1-score   support

           0       0.60      0.79      0.68       589
           1       0.61      0.58      0.60       430
           2       0.64      0.14      0.23       203

    accuracy                           0.61      1222
   macro avg       0.62      0.50      0.51      1222
weighted avg       0.61      0.61      0.58      1222



## Training a SVM classifier

In [ ]:
from sklearn.svm import LinearSVC

In [ ]:
svm_model = LinearSVC(
    C=0.5,
    class_weight="balanced",
    loss="hinge",
    max_iter=8000,
    tol=1e-4,
    dual="auto"
)

svm_model.fit(X_train, y_train)

LinearSVC(C=0.5, class_weight='balanced', loss='hinge', max_iter=8000)

In [ ]:
valid_preds = svm_model.predict(X_valid)

val_macro_f1 = f1_score(y_valid, valid_preds, average="macro")

print("\nValidation Macro F1:", val_macro_f1)
print("\nValidation Report:")
print(classification_report(y_valid, valid_preds))


Validation Macro F1: 0.5621837265570866

Validation Report:
              precision    recall  f1-score   support

           0       0.66      0.65      0.65       589
           1       0.64      0.59      0.61       430
           2       0.38      0.47      0.42       203

    accuracy                           0.60      1222
   macro avg       0.56      0.57      0.56      1222
weighted avg       0.61      0.60      0.60      1222



In [ ]:
test_preds = svm_model.predict(X_test)

test_macro_f1 = f1_score(y_test, test_preds, average="macro")

print("\nTest Macro F1:", test_macro_f1)
print("\nTest Report:")
print(classification_report(y_test, test_preds))



Test Macro F1: 0.6009130062454561

Test Report:
              precision    recall  f1-score   support

           0       0.68      0.66      0.67       589
           1       0.62      0.61      0.62       430
           2       0.48      0.55      0.51       203

    accuracy                           0.62      1222
   macro avg       0.60      0.61      0.60      1222
weighted avg       0.63      0.62      0.63      1222



## Decision tree classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=50,          # controls overfitting
    min_samples_split=50,
    min_samples_leaf=20,
    random_state=42,
    criterion="entropy",
)

dt_model.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=50, min_samples_leaf=20,
                       min_samples_split=50, random_state=42)

In [ ]:
valid_preds = dt_model.predict(X_valid)

val_macro_f1 = f1_score(y_valid, valid_preds, average="macro")

print("\nValidation Macro F1:", val_macro_f1)
print("\nValidation Report:")
print(classification_report(y_valid, valid_preds))


Validation Macro F1: 0.47202916737940886

Validation Report:
              precision    recall  f1-score   support

           0       0.58      0.69      0.63       589
           1       0.52      0.49      0.50       430
           2       0.38      0.23      0.28       203

    accuracy                           0.54      1222
   macro avg       0.49      0.47      0.47      1222
weighted avg       0.53      0.54      0.53      1222



In [ ]:
test_preds = dt_model.predict(X_test)

test_macro_f1 = f1_score(y_test, test_preds, average="macro")

print("\nTest Macro F1:", test_macro_f1)
print("\nTest Report:")
print(classification_report(y_test, test_preds))


Test Macro F1: 0.49826923076923074

Test Report:
              precision    recall  f1-score   support

           0       0.57      0.67      0.62       589
           1       0.50      0.47      0.49       430
           2       0.52      0.32      0.39       203

    accuracy                           0.54      1222
   macro avg       0.53      0.49      0.50      1222
weighted avg       0.54      0.54      0.53      1222



## create a random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,        # number of trees
    max_depth=None,          # allow deep trees
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",     # important for RF
    class_weight="balanced", # helps imbalance
    n_jobs=-1,               # use all CPU cores
    random_state=42
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', min_samples_leaf=2,
                       min_samples_split=5, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [ ]:
valid_preds = rf_model.predict(X_valid)

val_macro_f1 = f1_score(y_valid, valid_preds, average="macro")

print("\nValidation Macro F1:", val_macro_f1)
print("\nValidation Report:")
print(classification_report(y_valid, valid_preds))


Validation Macro F1: 0.5488943572440307

Validation Report:
              precision    recall  f1-score   support

           0       0.67      0.56      0.61       589
           1       0.59      0.62      0.60       430
           2       0.37      0.51      0.43       203

    accuracy                           0.57      1222
   macro avg       0.54      0.56      0.55      1222
weighted avg       0.59      0.57      0.58      1222



In [ ]:
test_preds = rf_model.predict(X_test)

test_macro_f1 = f1_score(y_test, test_preds, average="macro")

print("\nTest Macro F1:", test_macro_f1)
print("\nTest Report:")
print(classification_report(y_test, test_preds))


Test Macro F1: 0.582251755647526

Test Report:
              precision    recall  f1-score   support

           0       0.69      0.58      0.63       589
           1       0.60      0.64      0.62       430
           2       0.44      0.57      0.50       203

    accuracy                           0.60      1222
   macro avg       0.58      0.60      0.58      1222
weighted avg       0.61      0.60      0.60      1222



## MLP
#### Embedding → Mean Pool → MLP → Output

In [ ]:
vocab_size = tokenizer.vocab_size
embed_dim   = 128
num_classes = 3

In [ ]:
class MLPClassifier(L.LightningModule):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_classes,
        lr=1e-3,
        hidden_dim=256,
        dropout=0.3
    ):
        super().__init__()
        self.save_hyperparameters()

        # Embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # MLP Head
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, num_classes)
        )

        self.f1 = MulticlassF1Score(num_classes=num_classes)

    def forward(self, input_ids, attention_mask):
        embeds = self.embedding(input_ids)  # (B, L, D)

        # Mask padding
        mask = attention_mask.unsqueeze(-1)  # (B, L, 1)
        masked_embeds = embeds * mask

        # Mean pooling
        summed = masked_embeds.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        mean_pooled = summed / counts  # (B, D)

        logits = self.mlp(mean_pooled)  # (B, C)
        return logits

    def training_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("train_loss", loss)
        self.log("train_f1", self.f1(preds, batch["labels"]))

        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_f1", self.f1(preds, batch["labels"]), prog_bar=True)

    def test_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("test_loss", loss)
        self.log("test_f1", self.f1(preds, batch["labels"]))

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)



In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",
    mode="max",
    save_top_k=1,
    filename="best-mlp"
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

In [ ]:
model = MLPClassifier(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_classes=num_classes,
    lr=1e-3,
    hidden_dim=256,
    dropout=0.3
)

trainer = L.Trainer(
    max_epochs=20,
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_callback, early_stop_callback]
)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding │ Embedding         │  3.9 M │ train │     0 │
│ 1 │ mlp       │ Sequential        │ 66.3 K │ train │     0 │
│ 2 │ f1        │ MulticlassF1Score │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 4.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.0 M                                                                                                
Total estimated model params size (MB): 15                                                                         
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

In [ ]:
!find . -name "best-mlp.ckpt"

./lightning_logs/version_10/checkpoints/best-mlp.ckpt
./lightning_logs/version_6/checkpoints/best-mlp.ckpt
./lightning_logs/version_5/checkpoints/best-mlp.ckpt
./lightning_logs/version_1/checkpoints/best-mlp.ckpt
./lightning_logs/version_4/checkpoints/best-mlp.ckpt
./lightning_logs/version_2/checkpoints/best-mlp.ckpt
./lightning_logs/version_9/checkpoints/best-mlp.ckpt
./lightning_logs/version_0/checkpoints/best-mlp.ckpt
./lightning_logs/version_7/checkpoints/best-mlp.ckpt
./lightning_logs/version_8/checkpoints/best-mlp.ckpt
./lightning_logs/version_3/checkpoints/best-mlp.ckpt


In [ ]:
checkpoint_path = "./lightning_logs/version_10/checkpoints/best-mlp.ckpt"
model = MLPClassifier.load_from_checkpoint(checkpoint_path)
model.eval()
model.cuda()

MLPClassifier(
  (embedding): Embedding(30522, 128)
  (mlp): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
  (f1): MulticlassF1Score()
)

In [ ]:
test_dataset = TweetDataset(test_enc, test_df["Label"].values)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
trainer.test(model, test_loader)


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test_f1          │     0.570741593837738     │
│         test_loss         │    0.8525053262710571     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.8525053262710571, 'test_f1': 0.570741593837738}]

## CNN(conv1D)

In [ ]:
vocab_size = tokenizer.vocab_size
embed_dim   = 300
num_classes = 3
num_filters = 256

In [ ]:
class CNN1DClassifier(L.LightningModule):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_classes,
        lr=1e-3,
        num_filters=num_filters,
        kernel_sizes=(2, 3, 4, 5),
        dropout=0.5
    ):
        super().__init__()
        self.save_hyperparameters()

        # Embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embed_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        # Final classifier
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

        self.f1 = MulticlassF1Score(num_classes=num_classes)

    def forward(self, input_ids, attention_mask):
        # (B, L) → (B, L, D)
        embeds = self.embedding(input_ids)

        # Convert for Conv1D → (B, D, L)
        embeds = embeds.permute(0, 2, 1)

        conv_outputs = []

        for conv in self.convs:
            x = conv(embeds)        # (B, F, L')
            x = F.relu(x)
            x = F.max_pool1d(x, kernel_size=x.shape[2])  # Global max pool
            x = x.squeeze(2)        # (B, F)
            conv_outputs.append(x)

        # Concatenate filters
        x = torch.cat(conv_outputs, dim=1)  # (B, F * K)

        x = self.dropout(x)

        logits = self.fc(x)

        return logits

    def training_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("train_loss", loss)
        self.log("train_f1", self.f1(preds, batch["labels"]))

        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_f1", self.f1(preds, batch["labels"]), prog_bar=True)

    def test_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("test_loss", loss)
        self.log("test_f1", self.f1(preds, batch["labels"]))

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr,weight_decay=1e-4)

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",
    mode="max",
    save_top_k=1,
    filename="best-cnn"
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

In [ ]:
model = CNN1DClassifier(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_classes=3,
    lr=1e-3,
    num_filters=num_filters,
    kernel_sizes=(2,3,4,5),
    dropout=0.5
)

trainer = L.Trainer(
    max_epochs=20,
    accelerator="gpu",
    devices=1,
    deterministic=True,
    callbacks=[checkpoint_callback, early_stop_callback]
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding │ Embedding         │  9.2 M │ train │     0 │
│ 1 │ convs     │ ModuleList        │  1.1 M │ train │     0 │
│ 2 │ dropout   │ Dropout           │      0 │ train │     0 │
│ 3 │ fc        │ Linear            │  3.1 K │ train │     0 │
│ 4 │ f1        │ MulticlassF1Score │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 10.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.2 M                                                                                               
Total estimated model params size (MB): 40                                                                         
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

In [ ]:
!find . -name "best-cnn.ckpt"

./lightning_logs/version_12/checkpoints/best-cnn.ckpt
./lightning_logs/version_13/checkpoints/best-cnn.ckpt
./lightning_logs/version_17/checkpoints/best-cnn.ckpt
./lightning_logs/version_16/checkpoints/best-cnn.ckpt
./lightning_logs/version_20/checkpoints/best-cnn.ckpt
./lightning_logs/version_11/checkpoints/best-cnn.ckpt
./lightning_logs/version_15/checkpoints/best-cnn.ckpt
./lightning_logs/version_19/checkpoints/best-cnn.ckpt
./lightning_logs/version_14/checkpoints/best-cnn.ckpt


In [ ]:
checkpoint_path = "./lightning_logs/version_20/checkpoints/best-cnn.ckpt"
model = CNN1DClassifier.load_from_checkpoint(checkpoint_path)
model.eval()
model.cuda()

CNN1DClassifier(
  (embedding): Embedding(30522, 300)
  (convs): ModuleList(
    (0): Conv1d(300, 256, kernel_size=(2,), stride=(1,))
    (1): Conv1d(300, 256, kernel_size=(3,), stride=(1,))
    (2): Conv1d(300, 256, kernel_size=(4,), stride=(1,))
    (3): Conv1d(300, 256, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=1024, out_features=3, bias=True)
  (f1): MulticlassF1Score()
)

In [ ]:
test_dataset = TweetDataset(test_enc, test_df["Label"].values)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
trainer.test(model, test_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test_f1          │     0.600888192653656     │
│         test_loss         │    0.9074403643608093     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.9074403643608093, 'test_f1': 0.600888192653656}]

## creating a LSTM

In [ ]:
vocab_size = tokenizer.vocab_size
embed_dim   = 128
num_classes = 3

In [ ]:
class LSTMClassifier(L.LightningModule):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        num_classes,
        num_layers=2,
        bidirectional=True,
        dropout=0.3,
        lr=1e-3
    ):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # LSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0
        )

        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim

        # Classifier
        self.fc = nn.Linear(lstm_output_dim, num_classes)

        self.dropout = nn.Dropout(dropout)

        self.f1 = MulticlassF1Score(num_classes=num_classes)

    def forward(self, input_ids, attention_mask):
        # (B, L) → (B, L, D)
        embeds = self.embedding(input_ids)

        # LSTM output
        outputs, (hidden, cell) = self.lstm(embeds)

        # Use final hidden state
        if self.hparams.bidirectional:
            final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            final_hidden = hidden[-1]

        final_hidden = self.dropout(final_hidden)

        logits = self.fc(final_hidden)

        return logits

    def training_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("train_loss", loss)
        self.log("train_f1", self.f1(preds, batch["labels"]))

        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_f1", self.f1(preds, batch["labels"]), prog_bar=True)

    def test_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = F.cross_entropy(logits, batch["labels"])

        preds = torch.argmax(logits, dim=1)

        self.log("test_loss", loss)
        self.log("test_f1", self.f1(preds, batch["labels"]))

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)


In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",
    mode="max",
    save_top_k=1,
    filename="best-lstm"
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

In [ ]:
model = LSTMClassifier(
    vocab_size=vocab_size,
    embed_dim=128,
    hidden_dim=128,
    num_classes=3,
    num_layers=2,
    bidirectional=True,
    dropout=0.3,
    lr=1e-3
)

trainer = L.Trainer(
    max_epochs=20,
    accelerator="gpu",
    devices=1,
    deterministic=True,
    callbacks=[checkpoint_callback, early_stop_callback]
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding │ Embedding         │  3.9 M │ train │     0 │
│ 1 │ lstm      │ LSTM              │  659 K │ train │     0 │
│ 2 │ fc        │ Linear            │    771 │ train │     0 │
│ 3 │ dropout   │ Dropout           │      0 │ train │     0 │
│ 4 │ f1        │ MulticlassF1Score │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 4.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.6 M                                                                                                
Total estimated model params size (MB): 18                                                                         
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

In [ ]:
!find . -name "best-lstm.ckpt"

./lightning_logs/version_24/checkpoints/best-lstm.ckpt
./lightning_logs/version_25/checkpoints/best-lstm.ckpt
./lightning_logs/version_21/checkpoints/best-lstm.ckpt
./lightning_logs/version_26/checkpoints/best-lstm.ckpt
./lightning_logs/version_27/checkpoints/best-lstm.ckpt
./lightning_logs/version_22/checkpoints/best-lstm.ckpt
./lightning_logs/version_23/checkpoints/best-lstm.ckpt


In [ ]:
checkpoint_path = "./lightning_logs/version_27/checkpoints/best-lstm.ckpt"
model = LSTMClassifier.load_from_checkpoint(checkpoint_path)
model.eval()
model.cuda()

LSTMClassifier(
  (embedding): Embedding(30522, 128)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=256, out_features=3, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (f1): MulticlassF1Score()
)

In [ ]:
test_dataset = TweetDataset(test_enc, test_df["Label"].values)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
trainer.test(model, test_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test_f1          │    0.5482349395751953     │
│         test_loss         │    0.9855674505233765     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.9855674505233765, 'test_f1': 0.5482349395751953}]